In [ ]:
import gridlabd

In [ ]:
dir(gridlabd)

In [ ]:
gld = gridlabd.GridLabD()

In [ ]:
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld.set_working_directory(str(model_dir))


In [ ]:
gld.set_config_file("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf")

In [ ]:
gld.load_glm(["gridlabd", "./test_HVAC_balance.glm", "--verbose"])

In [ ]:
gld.set_time_step(86400)  # Set time step to 1 day in seconds

In [ ]:
gld.step()

In [ ]:
import json
# Get checkpoint as JSON string
checkpoint_json = gld.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

print("Checkpoint keys:", list(checkpoint_data.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data)}")

In [ ]:
list(checkpoint_data.keys())

In [ ]:
checkpoint_data['__preamble']

In [ ]:
checkpoint_data['clock']

In [ ]:
list(checkpoint_data['objects'])

In [ ]:
# Get current simulation time
status, current_time = gld.get_time()
print(f"Status: {status}")
print(f"Current simulation time: {current_time}")

In [ ]:
print(checkpoint_data['objects']['house'])

In [ ]:

gld.step()


In [ ]:
checkpoint_json = gld.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

In [ ]:
print(checkpoint_data['objects']['house'])

In [ ]:
# Don't call exit_gld() in notebooks - it crashes the kernel
# Just let Python clean up automatically
del gld

In [1]:
# Test set_time_step behavior
import gridlabd

# Create fresh instance
gld_test = gridlabd.GridLabD()
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld_test.set_working_directory(str(model_dir))
gld_test.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("Before set_time_step:")
status1, time1 = gld_test.get_time()
print(f"  Time: {time1}")

# Set time step to 1 day (86400 seconds)
result = gld_test.set_time_step(86400)
print(f"\nset_time_step(86400) returned: {result}")

# Step once
print("\nCalling step()...")
status2, sim_time = gld_test.step()
print(f"  Status: {status2}")
print(f"  Returned sim_time: {sim_time}")

# Check actual time
status3, time2 = gld_test.get_time()
print(f"  Actual time after step: {time2}")

# Parse times to see the difference
from datetime import datetime
dt1 = datetime.fromisoformat(time1)
dt2 = datetime.fromisoformat(time2)
diff = (dt2 - dt1).total_seconds()
print(f"\nTime difference: {diff} seconds ({diff/3600} hours, {diff/86400} days)")
print(f"Expected: 86400 seconds (1 day)")
print(f"Problem: Stepping by {diff} seconds instead of 86400!")


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests
Before set_time_step:
  Time: 2025-06-12T12:00:00

set_time_step(86400) returned: GLDErrorCode.SUCCESS

Calling step()...
Getting current time: 2025-06-12T12:00:00
Setting minimum simulation time step to: 86400 seconds
Stepping simulation forward
Simulation not initialized, attempting to initialize...

WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.
Simulation initialized successfully
Stepped from time 989064000.00 to 989107200.00
  Status: GLDErrorCode.SUCCESS
  Returned sim_time: 989107200.0
  Actual time after step: 2025-06-12T12:00:00

Time difference: 0.0 seconds (0.0 hours, 0.0 days)
Expected: 86400 seconds (1 day)
Problem: Stepping by 0.0 seconds instead of 86400!
Getting current time: 2025-06-12T12:00:00


## Testing the fix for get_time() after set_time_step()

In [1]:
# Test with fixed get_time() - RESTART KERNEL FIRST!
import gridlabd
from pathlib import Path
from datetime import datetime

# Create fresh instance
gld_fixed = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld_fixed.set_working_directory(str(model_dir))
gld_fixed.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("=== Testing FIXED get_time() ===\n")
status1, time1 = gld_fixed.get_time()
print(f"Initial time: {time1}")

# Set time step to 1 day
gld_fixed.set_time_step(86400)
print(f"Set time step to 86400 seconds (1 day)\n")

# Step once
print("Stepping...")
status2, sim_time = gld_fixed.step()

# Get actual time after step
status3, time2 = gld_fixed.get_time()
print(f"Time after step: {time2}\n")

# Parse timestamps (GridLAB-D format: "YYYY-MM-DD HH:MM:SS TZ")
# Remove timezone abbreviation for parsing
def parse_gld_time(time_str):
    # Split off timezone if present
    parts = time_str.rsplit(' ', 1)
    if len(parts) == 2 and parts[1] in ['PST', 'PDT', 'EST', 'EDT', 'CST', 'CDT', 'MST', 'MDT']:
        time_str = parts[0]
    return datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')

dt1 = parse_gld_time(time1)
dt2 = parse_gld_time(time2)
diff_seconds = (dt2 - dt1).total_seconds()

print(f"Time advanced: {diff_seconds} seconds = {diff_seconds/3600:.1f} hours = {diff_seconds/86400:.2f} days")
print(f"Expected: 86400 seconds (1 day)")

if abs(diff_seconds - 86400) < 1:  # Within 1 second
    print("\n✅ SUCCESS! get_time() now correctly reflects the simulation time after step()")
else:
    print(f"\n❌ Still broken: Only advanced {diff_seconds} seconds instead of 86400")


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests
=== Testing FIXED get_time() ===

Initial time: 2001-05-05 05:00:00 PDT
Set time step to 86400 seconds (1 day)

Stepping...
Time after step: 2001-05-06 05:00:00 PDT

Time advanced: 86400.0 seconds = 24.0 hours = 1.00 days
Expected: 86400 seconds (1 day)

✅ SUCCESS! get_time() now correctly reflects the simulation time after step()
Getting current time: 2001-05-05 05:00:00 PDT
Setting minimum simulation time step to: 86400 seconds
Stepping simulation forward
Simulation not initialized, attempting to initialize...

WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.
Simulation initialized successfully
Stepping from time 989064000.00 to target 989150400.00 (step size: 86400 seconds)
  Internal step: 989064000.00 -> 989107200.00
  Internal step: 989107200.00 -> 989150400.00
Completed step: advanced from 989064000.00 

## Summary of Findings

### ✅ What's Working:
1. **`set_time_step(86400)` is working** - It sets the minimum timestep to 1 day
2. **`get_time()` is now fixed** - It correctly returns the actual simulation time
3. **`step()` is working** - It advances the simulation correctly

### 🎯 The "Issue" (Actually Expected Behavior):

The simulation advances **12 hours** instead of **24 hours** because:

**GridLAB-D's `step()` advances to the NEXT EVENT, not by a fixed amount!**

- `set_time_step()` sets a *minimum* interval
- Objects in the model (HVAC, houses, etc.) can request updates at any time
- GridLAB-D stops at the earliest requested event time
- In this model, something needs to update every 12 hours (likely the HVAC system or climate data)

### 📝 How GridLAB-D Timesteps Work:

1. You call `set_time_step(86400)` → Sets minimum to 1 day
2. You call `step()` → GridLAB-D asks all objects "when do you need to update?"
3. Objects respond: "I need update in 12 hours" (HVAC), "24 hours" (other), etc.
4. GridLAB-D picks the **earliest** time: **12 hours**
5. Simulation advances by 12 hours

This is **correct behavior** for an event-driven simulation!

### 🔧 If You Want Fixed 1-Day Steps:

You'd need to call `step()` multiple times until you reach the desired time:

```python
target_time = current_time + 86400  # 1 day ahead
while global_clock < target_time:
    gld.step()
```

But this defeats the purpose of event-driven simulation!